# Notebook 11 — Adiabatic CZ with Time-Dependent $\Omega(t)$ and $\Delta(t)$

This notebook moves from constant-parameter driving to a simple **adiabatic / dressed trajectory**.

Instead of holding the drive and detuning fixed, we use smooth schedules:
- $\Omega(t)$ ramps up and down,
- $\Delta(t)$ sweeps during the interaction,

with the goal of:

1. reducing leakage,
2. keeping the system close to a dressed-state path,
3. accumulating controlled phase while returning closer to the computational basis.

This is the natural next step after Notebook 10: **shape the path, not just the endpoints**.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, qeye, tensor, sigmax, sesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Time-dependent schedules

In [ ]:
def omega_schedule(t, T, omega_max):
    x = np.pi * t / T
    return omega_max * np.sin(x) ** 2

def delta_schedule(t, T, delta0, delta1):
    s = t / T
    return delta0 + (delta1 - delta0) * 0.5 * (1 - np.cos(np.pi * s))


## Time-dependent Hamiltonian

In [ ]:
def adiabatic_hamiltonian_parts():
    H_omega = 0.5 * (sx1 + sx2)
    H_delta = -(n1 + n2)
    H_V = n1 * n2
    return H_omega, H_delta, H_V

H_omega, H_delta, H_V = adiabatic_hamiltonian_parts()

def build_time_dependent_hamiltonian(T, omega_max, delta0, delta1, V):
    def coeff_omega(t, args=None):
        return omega_schedule(t, T=T, omega_max=omega_max)

    def coeff_delta(t, args=None):
        return delta_schedule(t, T=T, delta0=delta0, delta1=delta1)

    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Effective-unitary helpers

In [ ]:
def run_adiabatic_evolution(psi0, T, omega_max, delta0, delta1, V, n_steps=300):
    H = build_time_dependent_hamiltonian(T=T, omega_max=omega_max, delta0=delta0, delta1=delta1, V=V)
    times = np.linspace(0.0, T, n_steps)
    result = sesolve(H, psi0, times)
    return result.states[-1]

def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)

def build_effective_unitary(T, omega_max, delta0, delta1, V):
    columns = []
    for psi0 in basis_states:
        psi_final = run_adiabatic_evolution(
            psi0,
            T=T,
            omega_max=omega_max,
            delta0=delta0,
            delta1=delta1,
            V=V,
        )
        columns.append(state_to_column(psi_final))
    return np.column_stack(columns)


## Gate metrics

In [ ]:
def process_fidelity(U_eff, U_target=U_cz):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def phase_corrected_target(U_eff):
    phi_ent = entangling_phase(U_eff)
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)

def phase_corrected_fidelity(U_eff):
    return process_fidelity(U_eff, U_target=phase_corrected_target(U_eff))

def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))


## Compare a few adiabatic candidates

In [ ]:
V = 4.0
cases = [
    ('fast ramp', 10.0, 1.0, 3.0, 0.5),
    ('moderate ramp', 20.0, 1.0, 3.0, 0.5),
    ('slower stronger ramp', 30.0, 1.5, 4.0, 0.5),
]

for label, T, omega_max, delta1, delta0 in cases:
    U_eff = build_effective_unitary(T=T, omega_max=omega_max, delta0=delta0, delta1=delta1, V=V)
    print(label)
    print(f'  T={T:.3f}, omega_max={omega_max:.3f}, delta0={delta0:.3f}, delta1={delta1:.3f}')
    print(f'  raw fidelity:             {process_fidelity(U_eff):.6f}')
    print(f'  phase-corrected fidelity: {phase_corrected_fidelity(U_eff):.6f}')
    print(f'  entangling phase / pi:    {entangling_phase(U_eff)/np.pi:.6f}')
    print(f'  leakage:                  {leakage_metric(U_eff):.6f}')
    print()


## Effective unitary for a dressed / adiabatic candidate

In [ ]:
T_demo = 20.0
omega_max_demo = 1.0
delta0_demo = 0.5
delta1_demo = 3.0

U_demo = build_effective_unitary(
    T=T_demo,
    omega_max=omega_max_demo,
    delta0=delta0_demo,
    delta1=delta1_demo,
    V=V,
)

plt.figure(figsize=(6, 5))
im = plt.imshow(np.abs(U_demo), origin='lower', aspect='equal')
plt.xticks(range(4), basis_labels)
plt.yticks(range(4), basis_labels)
plt.xlabel('Input basis state')
plt.ylabel('Output basis amplitude')
plt.title(r'$|U_{eff}|$ for an adiabatic candidate')
plt.colorbar(im, label='magnitude')
plt.tight_layout()
plt.show()


## Gate-time scan at fixed schedules

In [ ]:
T_vals = np.linspace(6.0, 40.0, 40)
omega_max = 1.0
delta0 = 0.5
delta1 = 3.0

raw_vals = []
pc_vals = []
phi_vals = []
leak_vals = []

for T in T_vals:
    U_eff = build_effective_unitary(
        T=T,
        omega_max=omega_max,
        delta0=delta0,
        delta1=delta1,
        V=V,
    )
    raw_vals.append(process_fidelity(U_eff))
    pc_vals.append(phase_corrected_fidelity(U_eff))
    phi_vals.append(entangling_phase(U_eff) / np.pi)
    leak_vals.append(leakage_metric(U_eff))


In [ ]:
plt.figure(figsize=(7.5, 4.8))
plt.plot(T_vals, raw_vals, label='raw fidelity')
plt.plot(T_vals, pc_vals, label='phase-corrected fidelity')
plt.xlabel('Total gate time T')
plt.ylabel('Fidelity')
plt.title('Adiabatic gate-time scan')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(T_vals, phi_vals)
plt.axhline(1.0, linestyle='--', label='ideal +1')
plt.axhline(-1.0, linestyle='--', color='gray', label='ideal -1')
plt.xlabel('Total gate time T')
plt.ylabel(r'Entangling phase / $\pi$')
plt.title('Entangling phase under adiabatic schedules')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(T_vals, leak_vals)
plt.xlabel('Total gate time T')
plt.ylabel('Leakage norm')
plt.title('Leakage under adiabatic schedules')
plt.tight_layout()
plt.show()


## 2D scan over $(T, \Delta_1)$

In [ ]:
delta1_vals = np.linspace(1.0, 5.0, 24)
T_grid = np.linspace(6.0, 40.0, 28)

raw_map = np.zeros((len(delta1_vals), len(T_grid)))
pc_map = np.zeros((len(delta1_vals), len(T_grid)))
phi_map = np.zeros((len(delta1_vals), len(T_grid)))
leak_map = np.zeros((len(delta1_vals), len(T_grid)))

omega_max = 1.0
delta0 = 0.5

for i, delta1 in enumerate(delta1_vals):
    for j, T in enumerate(T_grid):
        U_eff = build_effective_unitary(
            T=T,
            omega_max=omega_max,
            delta0=delta0,
            delta1=delta1,
            V=V,
        )
        raw_map[i, j] = process_fidelity(U_eff)
        pc_map[i, j] = phase_corrected_fidelity(U_eff)
        phi_map[i, j] = entangling_phase(U_eff) / np.pi
        leak_map[i, j] = leakage_metric(U_eff)


In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(raw_map, origin='lower', aspect='auto', extent=[T_grid[0], T_grid[-1], delta1_vals[0], delta1_vals[-1]])
plt.contour(T_grid, delta1_vals, raw_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Final detuning $\Delta_1$')
plt.title(r'Raw fidelity over $(T, \Delta_1)$')
plt.colorbar(im, label='raw fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(pc_map, origin='lower', aspect='auto', extent=[T_grid[0], T_grid[-1], delta1_vals[0], delta1_vals[-1]])
plt.contour(T_grid, delta1_vals, pc_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Final detuning $\Delta_1$')
plt.title(r'Phase-corrected fidelity over $(T, \Delta_1)$')
plt.colorbar(im, label='phase-corrected fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(phi_map, origin='lower', aspect='auto', extent=[T_grid[0], T_grid[-1], delta1_vals[0], delta1_vals[-1]], vmin=-1, vmax=1)
plt.contour(T_grid, delta1_vals, phi_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Final detuning $\Delta_1$')
plt.title(r'Entangling phase / $\pi$ over $(T, \Delta_1)$')
plt.colorbar(im, label=r'$\phi_{ent}/\pi$')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(leak_map, origin='lower', aspect='auto', extent=[T_grid[0], T_grid[-1], delta1_vals[0], delta1_vals[-1]])
plt.contour(T_grid, delta1_vals, leak_map, colors='white', linewidths=0.4)
plt.xlabel('Total gate time T')
plt.ylabel(r'Final detuning $\Delta_1$')
plt.title(r'Leakage over $(T, \Delta_1)$')
plt.colorbar(im, label='leakage')
plt.tight_layout()
plt.show()


## Best adiabatic candidate

In [ ]:
best_idx = np.unravel_index(np.argmax(pc_map), pc_map.shape)
best_delta1 = delta1_vals[best_idx[0]]
best_T = T_grid[best_idx[1]]

U_best = build_effective_unitary(
    T=best_T,
    omega_max=omega_max,
    delta0=delta0,
    delta1=best_delta1,
    V=V,
)

print(f'Best adiabatic candidate: T = {best_T:.4f}, Delta1 = {best_delta1:.4f}')
print(f'  raw fidelity:             {process_fidelity(U_best):.6f}')
print(f'  phase-corrected fidelity: {phase_corrected_fidelity(U_best):.6f}')
print(f'  entangling phase / pi:    {entangling_phase(U_best)/np.pi:.6f}')
print(f'  leakage:                  {leakage_metric(U_best):.6f}')


## Interpretation

This notebook tests the central dressed-gate idea:

> use a smooth trajectory to accumulate phase without strongly populating the lossy manifold.

Compared with the previous notebooks, this approach should be better at:
- reducing violent population transfer,
- smoothing the phase accumulation path,
- improving the tradeoff between leakage and controlled phase.

If the adiabatic idea is working, you should see:
- lower leakage valleys,
- smoother entangling-phase structure,
- stronger phase-corrected fidelity in slower ramps.


## Optional next steps

Natural next upgrades are:

- optimize both $\Omega_{max}$ and $(\Delta_0, \Delta_1)$ together,
- use smoother or asymmetric schedules,
- compare resonant, echo, dressed, and adiabatic protocols in one final summary notebook,
- move to gradient-based optimal control once the adiabatic baseline is established.
